# DD Autocorrelation Excess Analysis

**Computes bias-corrected lag-1 autocorrelation of dependency distance sequences across UD treebanks with three tree-linearization baselines (RPL, FHD, SOP).**

This notebook demonstrates:
1. Core DD autocorrelation and baseline linearization algorithms
2. DerSimonian-Laird random-effects meta-analysis across treebanks
3. OLS/Mixed-effects typological regression (spoken vs. written, case morphology)
4. Forest plot visualization of excess autocorrelation

The demo uses 50 pre-computed treebank results covering 24 language families, 4 word-order types, and both spoken/written modalities.

In [1]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# loguru — NOT on Colab, always install
_pip('loguru==0.7.3')

# Core packages (pre-installed on Colab, install locally to match Colab env)
# scipy 1.16.3 requires Python >=3.11 (Colab uses 3.12); use 1.15.3 on Python 3.10
if 'google.colab' not in sys.modules:
    import platform
    py_minor = int(platform.python_version_tuple()[1])
    scipy_ver = 'scipy==1.16.3' if py_minor >= 11 else 'scipy==1.15.3'
    _pip('numpy==2.0.2', 'pandas==2.2.2', scipy_ver,
         'matplotlib==3.10.0', 'statsmodels==0.14.6')


[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python3 -m pip install --upgrade pip



[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python3 -m pip install --upgrade pip


In [2]:
import json
import math
import os
import sys
import time
import warnings
from collections import defaultdict

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from loguru import logger
from scipy import stats

# Logging
logger.remove()
logger.add(sys.stdout, level="INFO", format="{time:HH:mm:ss}|{level:<7}|{message}")

1

In [3]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-c3cfa4-sequential-dependency-distance-anti-corr/main/experiment_iter3_dd_autocorr_exp/demo/mini_demo_data.json"

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception: pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f: return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [4]:
data = load_data()
logger.info(f"Loaded {len(data['datasets'][0]['examples'])} treebank results")

04:15:36|INFO   |Loaded 50 treebank results


In [5]:
# ---------------------------------------------------------------------------
# Configuration — tunable parameters
# ---------------------------------------------------------------------------
# Number of treebanks to use from loaded data (min=5 for meta-analysis, max=50)
N_TREEBANKS = 50  # original: 233 (full UD), demo uses 50

# Reference parameters used to generate the pre-computed results:
N_PERMS_REFERENCE = 100    # original: 100 permutations per sentence
MIN_TOKENS_AFTER_PUNCT = 20  # original: 20

# Minimum treebanks for each analysis step
MIN_TB_META = 3       # minimum for meta-analysis
MIN_TB_REGRESSION = 10  # minimum for regression
MIN_TB_FOREST = 3     # minimum for forest plot

## Core Algorithm Functions

These functions implement the DD autocorrelation computation and three baseline linearization strategies:
- **RPL** (Random Projective Linearization): randomly assigns children left/right of head
- **FHD** (Fixed Head-Direction): uses corpus-derived head-direction per deprel
- **SOP** (Sibling-Order-Preserving): uses corpus-derived sibling order templates

In [6]:
def filter_punctuation(head_array: list, deprel_array: list):
    """Remove punct tokens and reindex heads.
    Returns (new_head_array, new_deprel_array) both 1-indexed, 0=root.
    """
    n = len(head_array)
    non_punct_idx = [i for i in range(n) if deprel_array[i] != "punct"]
    if not non_punct_idx:
        return [], []
    old_to_new = {0: 0}
    for new_i, old_i in enumerate(non_punct_idx):
        old_to_new[old_i + 1] = new_i + 1
    new_heads, new_deprels = [], []
    for old_i in non_punct_idx:
        old_head = head_array[old_i]
        visited = set()
        while old_head != 0 and old_head not in old_to_new and old_head not in visited:
            visited.add(old_head)
            old_head = head_array[old_head - 1]
        new_heads.append(old_to_new.get(old_head, 0))
        new_deprels.append(deprel_array[old_i])
    return new_heads, new_deprels


def compute_dd_consecutive(head_array: list) -> list:
    """Compute DD sequence using consecutive positions 1..n.
    DD_i = |i - head(i)| for non-root tokens.
    """
    dd = []
    for i, h in enumerate(head_array):
        if h == 0:
            continue
        dd.append(abs((i + 1) - h))
    return dd


def lag1_autocorrelation(seq: list) -> float:
    """Standard lag-1 autocorrelation r1."""
    x = np.asarray(seq, dtype=np.float64)
    n = len(x)
    if n < 4:
        return np.nan
    xbar = x.mean()
    diffs = x - xbar
    denom = np.dot(diffs, diffs)
    if denom == 0:
        return np.nan
    numer = np.dot(diffs[:-1], diffs[1:])
    return numer / denom


def r1_plus(seq: list) -> float:
    """Bias-corrected r1+ = r1 + 1/n (Huitema & McKean 1991)."""
    r1 = lag1_autocorrelation(seq)
    if np.isnan(r1):
        return np.nan
    return r1 + 1.0 / len(seq)


compute_autocorrelation = r1_plus


def build_children_map(head_array: list) -> tuple:
    """Build parent -> children map. Returns (children_map, root_node)."""
    children = defaultdict(list)
    root_node = None
    for i, h in enumerate(head_array):
        node_id = i + 1
        children[h].append(node_id)
        if h == 0:
            root_node = node_id
    return dict(children), root_node


def rpl_linearize(children_map: dict, root: int, rng: np.random.RandomState) -> list:
    """Random Projective Linearization."""
    def _lin(node):
        kids = children_map.get(node, [])
        if not kids:
            return [node]
        left, right = [], []
        for kid in kids:
            if rng.random() < 0.5:
                left.append(kid)
            else:
                right.append(kid)
        rng.shuffle(left)
        rng.shuffle(right)
        result = []
        for kid in left:
            result.extend(_lin(kid))
        result.append(node)
        for kid in right:
            result.extend(_lin(kid))
        return result
    return _lin(root)


def dd_from_linearization(linear_order: list, head_array: list) -> list:
    """Compute DD from a linearized order."""
    pos_map = {}
    for new_pos, node in enumerate(linear_order, 1):
        pos_map[node] = new_pos
    dd = []
    for new_pos, node in enumerate(linear_order, 1):
        head_node = head_array[node - 1]
        if head_node == 0:
            continue
        head_new_pos = pos_map.get(head_node)
        if head_new_pos is None:
            continue
        dd.append(abs(new_pos - head_new_pos))
    return dd


logger.info("Core algorithm functions defined")

04:15:36|INFO   |Core algorithm functions defined


## Parse Pre-computed Treebank Results

Each example in the loaded data contains a treebank's input (language metadata) and output (aggregated autocorrelation statistics). We parse these into a `results` dict and `typology` dict matching the original pipeline format.

In [7]:
examples = data["datasets"][0]["examples"][:N_TREEBANKS]

# Parse into results dict (treebank_id -> aggregated stats) and typology dict
results = {}
typology = {}

for ex in examples:
    tb_id = ex["metadata_treebank_id"]
    inp = json.loads(ex["input"])
    out = json.loads(ex["output"])

    results[tb_id] = out
    typology[tb_id] = {
        "language_name": inp.get("language_name"),
        "wals_case_category": inp.get("wals_case_category"),
        "wals_word_order_label": inp.get("wals_word_order_label"),
        "language_family": ex.get("metadata_language_family", inp.get("language_family")),
        "modality": ex.get("metadata_modality", inp.get("modality")),
        "ud_case_proportion": inp.get("ud_case_proportion"),
        "iso639_3": inp.get("iso639_3"),
    }

logger.info(f"Parsed {len(results)} treebank results")
logger.info(f"Language families: {len(set(t.get('language_family') for t in typology.values()))}")
logger.info(f"Modalities: {set(t.get('modality') for t in typology.values())}")

04:15:36|INFO   |Parsed 50 treebank results


04:15:36|INFO   |Language families: 24


04:15:36|INFO   |Modalities: {'mixed', 'written', 'spoken'}


## Meta-Analysis (DerSimonian-Laird)

Random-effects meta-analysis pools excess autocorrelation across treebanks, accounting for heterogeneity via the DerSimonian-Laird estimator for between-study variance (tau-squared).

In [8]:
def _meta_dl(y: np.ndarray, v: np.ndarray):
    """DerSimonian-Laird random-effects meta-analysis (manual implementation)."""
    try:
        w = 1.0 / v
        w_sum = np.sum(w)
        pooled_fe = np.sum(w * y) / w_sum
        Q = np.sum(w * (y - pooled_fe) ** 2)
        K = len(y)
        C = w_sum - np.sum(w ** 2) / w_sum
        tau2 = max(0.0, (Q - (K - 1)) / C)
        w_re = 1.0 / (v + tau2)
        w_re_sum = np.sum(w_re)
        pooled_re = float(np.sum(w_re * y) / w_re_sum)
        se_re = float(1.0 / np.sqrt(w_re_sum))
        return pooled_re, se_re, float(tau2)
    except Exception as e:
        logger.error(f"  DL meta-analysis failed: {e}")
        return None, None, None


def run_meta_analysis(results: dict) -> dict:
    """Run random-effects meta-analysis for each excess measure."""
    logger.info("Running meta-analysis...")
    meta_output = {}

    for tier_name, excess_key, se_key in [
        ("excess_RPL", "mean_excess_rpl", "se_excess_rpl"),
        ("excess_FHD", "mean_excess_fhd", "se_excess_fhd"),
        ("excess_SOP", "mean_excess_sop", "se_excess_sop"),
    ]:
        tb_ids = [tb for tb in results if results[tb].get(excess_key) is not None
                  and results[tb].get(se_key) is not None
                  and results[tb][se_key] > 0]
        if len(tb_ids) < MIN_TB_META:
            logger.warning(f"  {tier_name}: only {len(tb_ids)} treebanks, skipping")
            meta_output[tier_name] = {"error": "too_few_treebanks", "K": len(tb_ids)}
            continue

        y = np.array([results[tb][excess_key] for tb in tb_ids])
        v = np.array([results[tb][se_key] ** 2 for tb in tb_ids])

        valid = np.isfinite(y) & np.isfinite(v) & (v > 0)
        y, v = y[valid], v[valid]
        K = len(y)

        if K < MIN_TB_META:
            meta_output[tier_name] = {"error": "too_few_valid", "K": K}
            continue

        pooled_est, pooled_se, tau2 = _meta_dl(y, v)
        if pooled_est is None:
            meta_output[tier_name] = {"error": "convergence_failure", "K": K}
            continue

        typical_v = float(np.median(v))
        I2 = float(tau2 / (tau2 + typical_v)) if (tau2 + typical_v) > 0 else 0.0
        z = pooled_est / pooled_se if pooled_se > 0 else 0.0
        p_value = float(2 * stats.norm.sf(abs(z)))
        ci_lo = float(pooled_est - 1.96 * pooled_se)
        ci_hi = float(pooled_est + 1.96 * pooled_se)

        V_pooled = 1.0 / np.sum(1.0 / v) if np.all(v > 0) else 0.0
        t_crit = float(stats.t.ppf(0.975, df=max(K - 2, 1)))
        pi_lo = float(pooled_est - t_crit * np.sqrt(tau2 + V_pooled))
        pi_hi = float(pooled_est + t_crit * np.sqrt(tau2 + V_pooled))

        meta_output[tier_name] = {
            "pooled_estimate": float(pooled_est),
            "pooled_se": float(pooled_se),
            "ci_95": [ci_lo, ci_hi],
            "tau_squared": float(tau2),
            "I_squared": float(I2),
            "prediction_interval": [pi_lo, pi_hi],
            "K_treebanks": int(K),
            "p_value": p_value,
            "prop_negative": float(np.mean(y < 0)),
            "median_effect": float(np.median(y)),
            "mean_effect": float(np.mean(y)),
        }
        logger.info(
            f"  {tier_name}: pooled={pooled_est:.4f}, CI=[{ci_lo:.4f},{ci_hi:.4f}], "
            f"I2={I2:.2f}, p={p_value:.2e}, prop_neg={np.mean(y<0):.2f}, K={K}"
        )
    return meta_output

meta_output = run_meta_analysis(results)

04:15:36|INFO   |Running meta-analysis...


04:15:36|INFO   |  excess_RPL: pooled=-0.1391, CI=[-0.1505,-0.1278], I2=0.95, p=2.04e-127, prop_neg=1.00, K=50


04:15:36|INFO   |  excess_FHD: pooled=-0.0902, CI=[-0.0982,-0.0822], I2=0.90, p=2.71e-108, prop_neg=0.98, K=50


04:15:36|INFO   |  excess_SOP: pooled=-0.0731, CI=[-0.0803,-0.0659], I2=0.87, p=3.57e-87, prop_neg=1.00, K=50


## Language-Level Sensitivity Meta-Analysis

Aggregates same-language treebanks before running meta-analysis, ensuring results are not driven by over-represented languages.

In [9]:
def run_language_level_meta(results: dict, typology: dict) -> dict:
    """Aggregate same-language treebanks before meta-analysis."""
    logger.info("Running language-level sensitivity meta-analysis...")
    lang_groups = defaultdict(list)
    for tb_id, res in results.items():
        iso = typology.get(tb_id, {}).get("iso639_3", tb_id)
        if res.get("mean_excess_rpl") is not None and res.get("se_excess_rpl") is not None:
            lang_groups[iso].append(res)

    lang_results = {}
    for iso, group in lang_groups.items():
        vals = [r["mean_excess_rpl"] for r in group if r["mean_excess_rpl"] is not None]
        ses = [r["se_excess_rpl"] for r in group if r["se_excess_rpl"] is not None and r["se_excess_rpl"] > 0]
        if vals and ses:
            mean_val = float(np.mean(vals))
            combined_se = float(1.0 / np.sqrt(sum(1.0 / (s ** 2) for s in ses)))
            lang_results[iso] = {"mean_excess_rpl": mean_val, "se_excess_rpl": combined_se}

    if len(lang_results) < MIN_TB_META:
        return {"error": "too_few_languages", "K": len(lang_results)}

    y = np.array([r["mean_excess_rpl"] for r in lang_results.values()])
    v = np.array([r["se_excess_rpl"] ** 2 for r in lang_results.values()])

    pooled, se, tau2 = _meta_dl(y, v)
    if pooled is None:
        return {"error": "convergence_failure"}

    result = {
        "pooled_estimate": float(pooled),
        "pooled_se": float(se),
        "tau_squared": float(tau2),
        "K_languages": len(lang_results),
        "p_value": float(2 * stats.norm.sf(abs(pooled / se))) if se > 0 else None,
    }
    logger.info(f"  Language-level: pooled={pooled:.4f}, K={len(lang_results)}, p={result['p_value']:.2e}")
    return result

lang_meta = run_language_level_meta(results, typology)

04:15:36|INFO   |Running language-level sensitivity meta-analysis...


04:15:36|INFO   |  Language-level: pooled=-0.1352, K=43, p=3.82e-117


## Typological Regression

OLS and mixed-effects regression testing whether excess autocorrelation varies by modality (spoken vs. written), morphological case richness, sentence length, and word order. Language family serves as random intercept in the mixed model.

In [10]:
def run_regression(results: dict, typology: dict) -> dict:
    """Run mixed-effects typological regression."""
    logger.info("Running typological regression...")
    reg_output = {}

    # Build DataFrame
    rows = []
    for tb_id, res in results.items():
        typo = typology.get(tb_id, {})
        if res.get("mean_excess_rpl") is None:
            continue
        rows.append({
            "treebank": tb_id,
            "excess_rpl": res["mean_excess_rpl"],
            "excess_fhd": res.get("mean_excess_fhd"),
            "excess_sop": res.get("mean_excess_sop"),
            "modality": typo.get("modality", "written"),
            "word_order": typo.get("wals_word_order_label"),
            "case_richness_wals": typo.get("wals_case_category"),
            "ud_case_proportion": typo.get("ud_case_proportion", 0.0),
            "language_family": typo.get("language_family", "Unknown"),
            "mean_sent_length": res.get("mean_dd_length"),
            "n_sentences": res["n_sentences"],
        })

    if len(rows) < MIN_TB_REGRESSION:
        logger.warning(f"Only {len(rows)} treebanks for regression, too few")
        return {"error": "too_few_treebanks", "n": len(rows)}

    df = pd.DataFrame(rows)
    df["is_spoken"] = (df["modality"] == "spoken").astype(int)
    df["case_measure"] = pd.to_numeric(df["ud_case_proportion"], errors="coerce").fillna(0.0)

    # WALS-UD cross-validation
    wals_mask = df["case_richness_wals"].notna()
    if wals_mask.sum() >= 10:
        from scipy.stats import spearmanr
        rho, p = spearmanr(
            df.loc[wals_mask, "case_richness_wals"].astype(float),
            df.loc[wals_mask, "case_measure"]
        )
        reg_output["wals_ud_spearman"] = {"rho": float(rho), "p": float(p)}
        logger.info(f"  WALS-UD case Spearman rho={rho:.3f}, p={p:.4f}")

    # Prepare regression data
    df_reg = df.dropna(subset=["excess_rpl", "language_family", "case_measure", "mean_sent_length"]).copy()
    family_counts = df_reg["language_family"].value_counts()
    valid_families = family_counts[family_counts >= 2].index.tolist()
    df_reg = df_reg[df_reg["language_family"].isin(valid_families)].copy()

    # Center continuous predictors
    case_mean = df_reg["case_measure"].mean()
    len_mean = df_reg["mean_sent_length"].mean()
    df_reg["case_c"] = df_reg["case_measure"] - case_mean
    df_reg["len_c"] = df_reg["mean_sent_length"] - len_mean
    reg_output["predictor_centering"] = {"case_mean": float(case_mean), "len_mean": float(len_mean)}

    import statsmodels.formula.api as smf
    import statsmodels.api as sm

    # OLS Model 1
    if len(df_reg) >= 15:
        try:
            X = df_reg[["is_spoken", "case_c", "len_c"]].copy()
            X = sm.add_constant(X)
            ols = sm.OLS(df_reg["excess_rpl"], X).fit()
            reg_output["model1_ols_summary"] = ols.summary().as_text()
            reg_output["model1_ols_params"] = {
                k: {"coef": float(v), "se": float(ols.bse[k]), "p": float(ols.pvalues[k])}
                for k, v in ols.params.items()
            }
            logger.info(f"  OLS Model 1 R2={ols.rsquared:.3f}")
        except Exception as e:
            logger.error(f"  OLS Model 1 failed: {e}")

    # MixedLM (family random intercepts)
    if len(df_reg) >= 15 and len(valid_families) >= 3:
        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                model1 = smf.mixedlm(
                    "excess_rpl ~ is_spoken + case_c + len_c",
                    data=df_reg, groups=df_reg["language_family"]
                )
                fit1 = model1.fit(reml=True, method="lbfgs")
            reg_output["model1_mixed_summary"] = fit1.summary().as_text()
            reg_output["model1_mixed_converged"] = fit1.converged
            reg_output["model1_mixed_params"] = {
                k: {"coef": float(v), "se": float(fit1.bse[k]), "p": float(fit1.pvalues[k])}
                for k, v in fit1.fe_params.items()
            }
            re_var = float(fit1.cov_re.iloc[0, 0]) if hasattr(fit1.cov_re, 'iloc') else 0.0
            reg_output["model1_mixed_re_variance"] = re_var
            logger.info(f"  MixedLM Model 1 converged: {fit1.converged}, RE var={re_var:.6f}")
        except Exception as e:
            logger.error(f"  MixedLM Model 1 failed: {e}")

    # Spoken vs Written comparison
    spoken = df_reg[df_reg["is_spoken"] == 1]
    written = df_reg[df_reg["is_spoken"] == 0]
    reg_output["n_spoken"] = len(spoken)
    reg_output["n_written"] = len(written)
    if len(spoken) >= 3 and len(written) >= 3:
        sp_mean = float(spoken["excess_rpl"].mean())
        wr_mean = float(written["excess_rpl"].mean())
        pooled_var = (spoken["excess_rpl"].var() + written["excess_rpl"].var()) / 2
        cohens_d = float((sp_mean - wr_mean) / np.sqrt(pooled_var)) if pooled_var > 0 else 0.0
        t_stat, t_p = stats.ttest_ind(spoken["excess_rpl"], written["excess_rpl"], equal_var=False)
        reg_output["spoken_written"] = {
            "spoken_mean": sp_mean, "written_mean": wr_mean,
            "cohens_d": cohens_d, "t_stat": float(t_stat), "p_value": float(t_p),
        }
        logger.info(f"  Spoken({len(spoken)}) mean={sp_mean:.4f} vs Written({len(written)}) mean={wr_mean:.4f}, d={cohens_d:.3f}")

    # FHD and SOP descriptive stats
    for measure_key, measure_name in [("excess_fhd", "FHD"), ("excess_sop", "SOP")]:
        col = measure_key
        subset = df_reg.dropna(subset=[col]) if col in df_reg.columns else pd.DataFrame()
        if len(subset) >= 10:
            reg_output[f"{measure_name}_descriptive"] = {
                "mean": float(subset[col].mean()),
                "se": float(subset[col].std() / np.sqrt(len(subset))),
                "n": len(subset),
            }

    return reg_output

reg_output = run_regression(results, typology)

04:15:36|INFO   |Running typological regression...


04:15:36|INFO   |  WALS-UD case Spearman rho=0.551, p=0.0011


04:15:39|INFO   |  OLS Model 1 R2=0.189


04:15:39|INFO   |  MixedLM Model 1 converged: True, RE var=0.001886


04:15:39|INFO   |  Spoken(13) mean=-0.1671 vs Written(16) mean=-0.1294, d=-0.843


## Forest Plots

Forest plots show per-treebank excess autocorrelation with 95% CIs, plus the pooled random-effects estimate (red dashed line with shaded CI band).

In [11]:
def generate_forest_plot(results: dict, measure_key: str, se_key: str,
                         meta_result: dict, title: str):
    """Generate a forest plot for the treebanks."""
    tb_ids = sorted(
        [tb for tb in results if results[tb].get(measure_key) is not None
         and results[tb].get(se_key) is not None and results[tb][se_key] > 0],
        key=lambda t: results[t]["n_sentences"], reverse=True
    )[:50]

    if len(tb_ids) < MIN_TB_FOREST:
        logger.warning(f"  Too few treebanks for forest plot: {len(tb_ids)}")
        return None

    estimates = [results[tb][measure_key] for tb in tb_ids]
    ses = [results[tb][se_key] for tb in tb_ids]
    ci_lo = [e - 1.96 * s for e, s in zip(estimates, ses)]
    ci_hi = [e + 1.96 * s for e, s in zip(estimates, ses)]

    fig, ax = plt.subplots(figsize=(10, max(8, len(tb_ids) * 0.3)))
    y_pos = list(range(len(tb_ids)))

    ax.errorbar(estimates, y_pos, xerr=[
        [e - lo for e, lo in zip(estimates, ci_lo)],
        [hi - e for e, hi in zip(estimates, ci_hi)]
    ], fmt='o', color='steelblue', ecolor='steelblue', elinewidth=1, capsize=2, markersize=4)

    pooled = meta_result.get("pooled_estimate")
    if pooled is not None:
        pooled_ci = meta_result.get("ci_95", [pooled, pooled])
        ax.axvspan(pooled_ci[0], pooled_ci[1], alpha=0.2, color='red')
        ax.axvline(pooled, color='red', linestyle='--', linewidth=1, label=f'Pooled={pooled:.4f}')

    ax.axvline(0, color='black', linestyle='-', linewidth=0.5)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(tb_ids, fontsize=6)
    ax.set_xlabel(title)
    ax.set_title(f"Forest Plot: {title} ({len(tb_ids)} treebanks)")
    ax.legend(fontsize=8)
    ax.invert_yaxis()
    plt.tight_layout()
    return fig


# Generate forest plots for all three measures
for measure, se_key, meta_key, title in [
    ("mean_excess_rpl", "se_excess_rpl", "excess_RPL", "Excess Autocorrelation (RPL baseline)"),
    ("mean_excess_fhd", "se_excess_fhd", "excess_FHD", "Excess Autocorrelation (FHD baseline)"),
    ("mean_excess_sop", "se_excess_sop", "excess_SOP", "Excess Autocorrelation (SOP baseline)"),
]:
    meta_res = meta_output.get(meta_key, {})
    fig = generate_forest_plot(results, measure, se_key, meta_res, title)
    if fig is not None:
        plt.show()
        plt.close(fig)

## Results Summary

Key findings: pooled excess autocorrelation across baselines, spoken vs. written comparison, and hierarchy |RPL| > |FHD| > |SOP|.

In [12]:
# --- Meta-analysis summary table ---
print("=" * 70)
print("META-ANALYSIS RESULTS (DerSimonian-Laird Random Effects)")
print("=" * 70)
print(f"{'Measure':<12} {'Pooled':>9} {'95% CI':>20} {'p-value':>12} {'I2':>6} {'K':>4} {'%neg':>6}")
print("-" * 70)
for key in ["excess_RPL", "excess_FHD", "excess_SOP"]:
    m = meta_output.get(key, {})
    if "pooled_estimate" in m:
        ci = m["ci_95"]
        print(f"{key:<12} {m['pooled_estimate']:>9.4f} [{ci[0]:>8.4f}, {ci[1]:>7.4f}] {m['p_value']:>12.2e} {m['I_squared']:>5.2f} {m['K_treebanks']:>4d} {m['prop_negative']:>5.1%}")
    else:
        print(f"{key:<12} {'N/A':>9} (error: {m.get('error', 'unknown')})")

# --- Hierarchy check ---
print("\n" + "=" * 70)
print("BASELINE HIERARCHY CHECK: |RPL| > |FHD| > |SOP|")
print("=" * 70)
rpl_abs = abs(meta_output.get("excess_RPL", {}).get("pooled_estimate", 0))
fhd_abs = abs(meta_output.get("excess_FHD", {}).get("pooled_estimate", 0))
sop_abs = abs(meta_output.get("excess_SOP", {}).get("pooled_estimate", 0))
print(f"|RPL| = {rpl_abs:.4f},  |FHD| = {fhd_abs:.4f},  |SOP| = {sop_abs:.4f}")
print(f"Hierarchy holds: {rpl_abs > fhd_abs > sop_abs}")

# --- Spoken vs Written ---
sw = reg_output.get("spoken_written", {})
if sw:
    print("\n" + "=" * 70)
    print("SPOKEN vs WRITTEN COMPARISON")
    print("=" * 70)
    print(f"Spoken  mean excess_rpl: {sw['spoken_mean']:.4f}  (n={reg_output.get('n_spoken', '?')})")
    print(f"Written mean excess_rpl: {sw['written_mean']:.4f}  (n={reg_output.get('n_written', '?')})")
    print(f"Cohen's d = {sw['cohens_d']:.3f},  t = {sw['t_stat']:.3f},  p = {sw['p_value']:.4f}")

# --- Language-level sensitivity ---
if "pooled_estimate" in lang_meta:
    print("\n" + "=" * 70)
    print("LANGUAGE-LEVEL SENSITIVITY")
    print("=" * 70)
    print(f"Pooled (language-aggregated): {lang_meta['pooled_estimate']:.4f}")
    print(f"K languages: {lang_meta['K_languages']},  p = {lang_meta['p_value']:.2e}")

# --- Bar chart of pooled estimates ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Pooled estimates by baseline
baselines = []
pooled_vals = []
ci_errs = []
for key, label in [("excess_RPL", "RPL"), ("excess_FHD", "FHD"), ("excess_SOP", "SOP")]:
    m = meta_output.get(key, {})
    if "pooled_estimate" in m:
        baselines.append(label)
        pooled_vals.append(m["pooled_estimate"])
        ci_errs.append(1.96 * m["pooled_se"])

colors = ['#2196F3', '#FF9800', '#4CAF50']
axes[0].barh(baselines, pooled_vals, xerr=ci_errs, color=colors[:len(baselines)],
             edgecolor='black', linewidth=0.5, capsize=5)
axes[0].axvline(0, color='black', linewidth=0.5)
axes[0].set_xlabel("Pooled Excess Autocorrelation")
axes[0].set_title("Pooled Effect by Baseline (95% CI)")
axes[0].invert_yaxis()

# Plot 2: Distribution of excess_rpl across treebanks
excess_vals = [r["mean_excess_rpl"] for r in results.values()
               if r.get("mean_excess_rpl") is not None]
axes[1].hist(excess_vals, bins=20, color='steelblue', edgecolor='black', alpha=0.7)
axes[1].axvline(0, color='black', linewidth=1, linestyle='-')
pooled_rpl = meta_output.get("excess_RPL", {}).get("pooled_estimate")
if pooled_rpl is not None:
    axes[1].axvline(pooled_rpl, color='red', linewidth=2, linestyle='--', label=f'Pooled={pooled_rpl:.4f}')
    axes[1].legend()
axes[1].set_xlabel("Excess Autocorrelation (RPL)")
axes[1].set_ylabel("Count")
axes[1].set_title(f"Distribution of Excess r1+ (RPL) across {len(excess_vals)} treebanks")

plt.tight_layout()
plt.show()

print("\nDemo complete!")

META-ANALYSIS RESULTS (DerSimonian-Laird Random Effects)
Measure         Pooled               95% CI      p-value     I2    K   %neg
----------------------------------------------------------------------
excess_RPL     -0.1391 [ -0.1505, -0.1278]    2.04e-127  0.95   50 100.0%
excess_FHD     -0.0902 [ -0.0982, -0.0822]    2.71e-108  0.90   50 98.0%
excess_SOP     -0.0731 [ -0.0803, -0.0659]     3.57e-87  0.87   50 100.0%

BASELINE HIERARCHY CHECK: |RPL| > |FHD| > |SOP|
|RPL| = 0.1391,  |FHD| = 0.0902,  |SOP| = 0.0731
Hierarchy holds: True

SPOKEN vs WRITTEN COMPARISON
Spoken  mean excess_rpl: -0.1671  (n=13)
Written mean excess_rpl: -0.1294  (n=16)
Cohen's d = -0.843,  t = -2.319,  p = 0.0290

LANGUAGE-LEVEL SENSITIVITY
Pooled (language-aggregated): -0.1352
K languages: 43,  p = 3.82e-117

Demo complete!
